# 03 · Experiments E1–E6 (results for the report)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MR-SAIRAM-7/tokenEdge/blob/main/notebooks/03_experiments.ipynb)

| ID | Question |
|---|---|
| E1 | Does ML beat the SMA baseline after costs on held-out data? |
| E2 | At what cost level does each strategy's edge disappear? |
| E3 | Does triple-barrier labeling beat fixed-horizon labeling? *(trains models)* |
| E4 | How much do look-ahead leaks inflate a backtest? *(trains models)* |
| E5 | Does the best strategy survive the Deflated Sharpe correction? |
| E6 | Does higher classification accuracy mean higher Sharpe? |

E1, E2, E5 and E6 use the models in `models/`. Train them first with notebooks 01 and 02 (or upload your model folders to `models/` in this Colab session). Tables are saved to `docs/results/`.

In [ ]:
# --- Setup: works in Google Colab and locally (run from the repo or its notebooks/ folder)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/tokenEdge"):
        !git clone -q https://github.com/MR-SAIRAM-7/tokenEdge.git /content/tokenEdge
    %cd /content/tokenEdge
    !pip install -q -e ".[train]"
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
import tokenedge
print("TokenEdge", tokenedge.__version__)

> If pip upgrades numpy or pandas, Colab may ask to restart. Use **Runtime → Restart session** and run the cells again (the clone and install are skipped the second time).

In [ ]:
import pandas as pd
from pathlib import Path
from tokenedge.ml.registry import get_registry
from tokenedge.experiments import (DEFAULT_MARKETS, e1_main_comparison, e2_cost_sensitivity,
                                   e3_label_ablation, e4_lookahead_bias, e6_accuracy_vs_profit)
RESULTS = Path("docs/results"); RESULTS.mkdir(parents=True, exist_ok=True)
print("registered models:", [m["model_id"] for m in get_registry().list()])
MARKETS = DEFAULT_MARKETS  # [(symbol, source, interval), ...]

## E1 + E5 · Main comparison with Deflated Sharpe

In [ ]:
e1, runs = e1_main_comparison(MARKETS)
e1.to_csv(RESULTS / "e1_main_comparison.csv", index=False)
for run in runs:
    print(f"\n{run.market[0]} {run.market[2]}:", run.report["verdict"])
e1.round(3)

## E2 · Cost sensitivity

In [ ]:
import matplotlib.pyplot as plt
e2 = e2_cost_sensitivity(runs)
e2.to_csv(RESULTS / "e2_cost_sensitivity.csv", index=False)
colors = {"sma_crossover": "#2a78d6", "xgboost": "#eb6834", "lstm": "#1baf7a", "buy_hold": "#6250d6"}
markets = e2["market"].unique()
fig, axes = plt.subplots(1, len(markets), figsize=(4 * len(markets), 3.2), squeeze=False)
for ax, mk in zip(axes[0], markets):
    for strat, g in e2[e2.market == mk].groupby("strategy"):
        ax.plot(g["bps_per_side"], g["sharpe"], marker="o", label=strat, color=colors.get(strat))
    ax.axhline(0, color="gray", lw=0.8); ax.set_title(mk); ax.set_xlabel("bps per side")
axes[0][0].set_ylabel("Sharpe"); axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## E6 · Accuracy vs profitability

In [ ]:
e6 = e6_accuracy_vs_profit(runs)
e6.to_csv(RESULTS / "e6_accuracy_vs_profit.csv", index=False)
e6.round(3)

## E3 · Labeling ablation (trains two XGBoost models, ~minutes)

In [ ]:
e3 = e3_label_ablation("SPY", "yfinance", "1d", n_trials=20)
e3.to_csv(RESULTS / "e3_label_ablation.csv", index=False)
e3.round(3)

## E4 · How much does look-ahead bias inflate results?

In [ ]:
e4 = e4_lookahead_bias("SPY", "yfinance", "1d")
e4.to_csv(RESULTS / "e4_lookahead_bias.csv", index=False)
e4.round(3)

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("tokenedge_results", "zip", RESULTS)
    files.download("tokenedge_results.zip")